# FSOT-2.1-Neural on Kaggle — EEG Emotion Features + Morse Chew/Query

**Author / theory:** Damian Arthur Palumbo — Fluid Spacetime Omni-Theory  
**Theory authority:** [FSOT-2.1-Lean](https://github.com/dappalumbo91/FSOT-2.1-Lean)  
**This notebook + FSOT code license:** Apache-2.0  

### Data inputs (third-party — not claimed as FSOT IP)
Add Data → **`birdy654/eeg-brainwave-dataset-feeling-emotions`**  
(Jordan J. Bird et al. style EEG *feature* set; labels POSITIVE / NEUTRAL / NEGATIVE.)

Optional second dataset: upload **`fsot-2-1-neural-kaggle.zip`** (our code package only).

### What this is / is not
- **Is:** FSOT micro-substrate demo — emotion EEG features → drive priors; literature Morse retrieval Q&A  
- **Is not:** a medical device; not a re-host of the emotions CSV; not a giant free-parameter LLM  

Industry-standard path: attach other Kaggle EEG/NLP datasets the same way and re-run the cells.

## 0. Environment + paths (Kaggle vs local)

In [ ]:
import os, sys, json, math, re, zipfile
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
print("ON_KAGGLE", ON_KAGGLE)

if ON_KAGGLE:
    INPUT = Path("/kaggle/input")
    WORK = Path("/kaggle/working")
else:
    # Local dev fallback
    INPUT = Path(r"I:/fsot nuron/data/eeg")
    WORK = Path(r"I:/fsot nuron/artifacts/kaggle_sim")
    WORK.mkdir(parents=True, exist_ok=True)
    ROOT_LOCAL = Path(r"I:/fsot nuron")
    if str(ROOT_LOCAL) not in sys.path:
        sys.path.insert(0, str(ROOT_LOCAL))

# Find emotions.csv in any attached input
def find_emotions_csv(root: Path):
    cands = list(root.rglob("emotions.csv")) if root.exists() else []
    # also common names
    if not cands and root.exists():
        cands = [p for p in root.rglob("*.csv") if "emotion" in p.name.lower() or p.name.lower()=="emotions.csv"]
    return cands[0] if cands else None

EMO_CSV = find_emotions_csv(INPUT)
print("emotions.csv:", EMO_CSV)

# Optional: unpack FSOT code package if attached as a dataset
FSOT_CODE = None
if ON_KAGGLE:
    for z in INPUT.rglob("fsot-2-1-neural-kaggle.zip"):
        dest = WORK / "fsot_code"
        dest.mkdir(exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        FSOT_CODE = dest
        sys.path.insert(0, str(dest))
        print("Unpacked FSOT code package:", dest)
        break
    # or already-extracted folder containing fsot_nuron/
    if FSOT_CODE is None:
        for p in INPUT.rglob("fsot_nuron"):
            if p.is_dir():
                sys.path.insert(0, str(p.parent))
                FSOT_CODE = p.parent
                print("Found fsot_nuron at", FSOT_CODE)
                break

USE_PACKAGE = False
try:
    import fsot_nuron
    USE_PACKAGE = True
    print("fsot_nuron package available:", fsot_nuron.__file__ if hasattr(fsot_nuron,'__file__') else fsot_nuron)
except Exception as e:
    print("Package not importable yet — using in-notebook compact core.", e)

## 1. Compact FSOT core (always available on Kaggle)

Self-contained seed scalar + ITU Morse + emotion stats + tiny reservoir chew/query.  
If full `fsot_nuron` package is attached, later cells can use it for richer features.

In [ ]:
import numpy as np
import pandas as pd

# --- Seeds / scalar (FSOT zero free-parameter spine, float form) ---
PHI, E, PI, GAMMA = 1.618033988749895, 2.718281828459045, math.pi, 0.5772156649015329
ALPHA = 8.082937414140405e-4
PSI_CON, ETA_EFF = 0.6321205588285577, 0.46694220692425986
B_IN, C_EFF, P_VAR = 0.7879407922764435, 0.9577022026205613, 0.9579871226722757
P_NEW, C_FACTOR, K = 0.30030227667037146, 0.287600151819184, 0.42022166416069665

def fsot_S(N=4.0, P=3.0, D_eff=13.0, recent_hits=0.0, delta_psi=0.1, rho=1.0, observed=True):
    growth = math.exp(ALPHA * (1.0 - recent_hits / max(N, 1e-6)) * GAMMA / PHI)
    base = (N * P / math.sqrt(max(D_eff, 1e-6))) * math.cos((PSI_CON + delta_psi) / ETA_EFF)
    base *= math.exp(-ALPHA * recent_hits / max(N, 1e-6) + rho + B_IN * delta_psi) * (1.0 + growth * C_EFF)
    t1 = base * (1.0 + P_NEW * math.log(max(D_eff, 1e-6) / 25.0))
    if observed:
        t1 *= math.exp(C_FACTOR * P_VAR) * math.cos(delta_psi + P_VAR)
    return max(-3.0, min(3.0, K * (t1 + 1.0)))  # +1 as simplified t2 scale

# --- ITU Morse (letters + digits) ---
MORSE = {
    'A':'.-','B':'-...','C':'-.-.','D':'-..','E':'.','F':'..-.','G':'--.','H':'....','I':'..','J':'.---',
    'K':'-.-','L':'.-..','M':'--','N':'-.','O':'---','P':'.--.','Q':'--.-','R':'.-.','S':'...','T':'-',
    'U':'..-','V':'...-','W':'.--','X':'-..-','Y':'-.--','Z':'--..',
    '0':'-----','1':'.----','2':'..---','3':'...--','4':'....-','5':'.....','6':'-....','7':'--...','8':'---..','9':'----.',
}
REV = {v:k for k,v in MORSE.items()}

def encode_morse(text: str) -> str:
    words = []
    for w in text.upper().split():
        letters = [MORSE[c] for c in w if c in MORSE]
        if letters:
            words.append(' '.join(letters))
    return ' / '.join(words)

def decode_morse(m: str) -> str:
    out = []
    for w in m.split(' / '):
        out.append(''.join(REV.get(t, '?') for t in w.split() if t))
    return ' '.join(out)

def text_to_units(text: str):
    stream = []
    words = text.upper().split()
    for wi, w in enumerate(words):
        if wi: stream += [0]*7
        first = True
        for ch in w:
            if ch not in MORSE: continue
            if not first: stream += [0]*3
            first = False
            for ei, el in enumerate(MORSE[ch]):
                if ei: stream.append(0)
                stream += [1]*(1 if el=='.' else 3)
    return stream

print('FSOT S smoke', round(fsot_S(), 4))
print('Morse RT', decode_morse(encode_morse('FSOT 21')) == 'FSOT 21')

## 2. Load industry EEG emotion features (Kaggle input)

In [ ]:
assert EMO_CSV is not None, (
    "Attach Kaggle dataset birdy654/eeg-brainwave-dataset-feeling-emotions "
    "(Add Data). Locally ensure emotions.csv is under the INPUT path."
)

df = pd.read_csv(EMO_CSV)
cols = list(df.columns)
if cols[0].startswith('#'):
    cols[0] = cols[0].lstrip('#').strip()
    df.columns = cols
label_col = 'label' if 'label' in df.columns else df.columns[-1]
y = df[label_col].astype(str).str.upper()
X = df.drop(columns=[label_col]).select_dtypes(include=[np.number])
X = X.dropna(axis=1, how='all')
if X.shape[1] > 256:
    prefer = [c for c in X.columns if str(c).startswith('mean_') or str(c).startswith('fft_')]
    X = X[prefer[:256]] if len(prefer) >= 64 else X.iloc[:, :256]

print('shape', df.shape, 'features_used', X.shape[1])
print('labels', y.value_counts().to_dict())

row_energy = X.abs().mean(axis=1)
energy_cv = float(row_energy.std() / (row_energy.mean() + 1e-12))
print('sample_energy_cv', round(energy_cv, 4))

by = {}
for lab in sorted(y.unique()):
    sub = X[y == lab]
    by[lab] = {
        'n': int(len(sub)),
        'mean_abs': float(sub.abs().mean().mean()),
        'std_mean': float(sub.std(axis=0).mean()),
        'energy_mean': float(sub.abs().mean(axis=1).mean()),
    }
print(json.dumps(by, indent=2))

# Emotion → FSOT drive scales
neu_e = by.get('NEUTRAL', {}).get('energy_mean') or 1.0
templates = {
    'POSITIVE': {'fi_stim_scale': min(1.4, 1.0 + 0.25*((by.get('POSITIVE',{}).get('energy_mean',neu_e)/neu_e-1)+0.2)),
                 'fire_thr_delta': -0.06, 'adapt_step_scale': 0.85, 'isi_jitter': 0.15},
    'NEUTRAL':  {'fi_stim_scale': 1.0, 'fire_thr_delta': 0.0, 'adapt_step_scale': 1.0, 'isi_jitter': 0.10},
    'NEGATIVE': {'fi_stim_scale': min(1.2, 0.95+0.15*(by.get('NEGATIVE',{}).get('std_mean',1)/(by.get('NEUTRAL',{}).get('std_mean',1) or 1))),
                 'fire_thr_delta': 0.05, 'adapt_step_scale': min(2.0, 1.2+0.5*energy_cv),
                 'isi_jitter': min(0.6, 0.25+0.4*energy_cv), 'silence_fraction': 0.05},
}
print('FSOT emotion templates')
print(json.dumps(templates, indent=2))

(WORK / 'kaggle_emotions_stats.json').write_text(json.dumps({
    'source_csv': str(EMO_CSV),
    'n_samples': int(df.shape[0]),
    'labels': y.value_counts().to_dict(),
    'energy_cv': energy_cv,
    'by_label': by,
    'templates': templates,
    'dataset_ref': 'birdy654/eeg-brainwave-dataset-feeling-emotions',
}, indent=2), encoding='utf-8')
print('wrote', WORK / 'kaggle_emotions_stats.json')

In [ ]:
import matplotlib.pyplot as plt

def emotion_drive(label, length=200, seed=0):
    tmpl = templates.get(label.upper(), templates['NEUTRAL'])
    rng = np.random.default_rng(seed)
    t = np.arange(length)
    amp = float(tmpl['fi_stim_scale'])
    jitter = float(tmpl['isi_jitter'])
    thr = float(tmpl.get('fire_thr_delta', 0.0))
    sig = amp * (0.35*np.sin(2*np.pi*t/25) + 0.2*np.sin(2*np.pi*t/80))
    sig = sig + jitter * rng.standard_normal(length)
    sig = 0.45 + 0.35*np.tanh(sig) - 0.1*thr
    return np.clip(sig, 0.05, 1.15)

plt.figure(figsize=(9,3))
for lab in ['NEGATIVE','NEUTRAL','POSITIVE']:
    if lab in y.unique():
        plt.plot(emotion_drive(lab), label=lab, alpha=0.9)
plt.legend(); plt.title('Emotion-conditioned FSOT drive'); plt.xlabel('t'); plt.ylabel('stim'); plt.tight_layout(); plt.show()

## 3. Tiny FSOT reservoir + Morse memory chew / query

Chews built-in FSOT literature snippets (in-notebook) so the kernel runs **without** shipping a corpus.  
On your machine / full package you can chew larger archives.

In [ ]:
CORPUS = [
    ("FSOT_scalar", "Fluid Spacetime Omni-Theory FSOT derives physical and biological relations from seeds pi e phi gamma with zero free parameters on the theory path. The scalar S equals K times T1 plus T2 plus T3."),
    ("FSOT_morse", "Morse-trinary encodes language for an FSOT neural reservoir. Plus one is emergent dash, minus one is damped dot, zero is separator. The system can ingest Shakespeare and regurgitate meaningful phrases."),
    ("FSOT_neuron", "FSOT active neurons use phase, refractory dynamics, and trinary state minus one damped zero stable plus one emergent. Adaptation and ISI can be locked to Allen cell type statistics."),
    ("FSOT_emotion_eeg", "EEG emotion feature datasets labeled POSITIVE NEUTRAL NEGATIVE provide irregularity and energy statistics that modulate FSOT drive and lesion priors for substrate boundary tests."),
    ("FSOT_lean", "Formal verification of FSOT 2.1 lives in the FSOT-2.1-Lean repository. This neural substrate applies the theory; it does not re-derive the multi-domain Lean panels."),
    ("FSOT_consensus", "FSOT consensus attention aggregates without softmax using collapse gated trinary similarity for long context substrate operators on GPU."),
]

def run_reservoir_on_text(text, steps_cap=300):
    units = text_to_units(text)[:steps_cap]
    if not units:
        units = [0,1,0]
    # minimal stateful S trace
    S_hist, fire = [], []
    phase, ref, S = 0.05, 0, 0.46
    spikes = 0
    for u in units:
        stim = 0.12 + 0.78*u
        if ref > 0:
            ref -= 1
            rh, dpsi = 2.0, phase*0.4
        else:
            rh = 1.0 if stim > 0.3 else 0.0
            dpsi = phase*0.85 + 0.05 + stim*0.04
        S = fsot_S(recent_hits=rh, delta_psi=dpsi, rho=1.0+0.1*(S-0.46)+0.4*stim)
        phase = (phase + 0.02 + 0.05*abs(S)) % (2*math.pi)
        thr = 1.05 - 0.4*stim
        fired = (ref==0) and (S > thr)
        if fired:
            ref = 8
            spikes += 1
            phase = 0.0
        S_hist.append(S); fire.append(1 if fired else 0)
    S_hist = np.array(S_hist)
    # fingerprint
    tern = np.where(S_hist>0.4, 1, np.where(S_hist<-0.05, -1, 0))
    n = len(tern)
    fp = [
        float((tern<0).mean()), float((tern==0).mean()), float((tern>0).mean()),
        float(S_hist.mean()), float(S_hist.std()), float(spikes)/(max(n,1)/1000.0)
    ]
    return {'fp': fp, 'S_mean': float(S_hist.mean()), 'rate': float(spikes)/(max(n,1)/1000.0), 'morse': encode_morse(text)}

memory = []
for src, text in CORPUS:
    r = run_reservoir_on_text(text)
    memory.append({'source': src, 'text': text, **r})
print('memory chunks', len(memory))

def cosine(a,b):
    a,b = np.array(a), np.array(b)
    return float(a@b / ((np.linalg.norm(a)*np.linalg.norm(b))+1e-9))

def query(question, top_k=3):
    q = run_reservoir_on_text(question)
    qwords = {w for w in re.findall(r'[A-Za-z0-9]+', question.upper()) if len(w)>2}
    stop = {'WHAT','IS','THE','HOW','DOES','ARE','AND','FOR','WITH'}
    qwords -= stop
    scored = []
    for m in memory:
        mwords = set(re.findall(r'[A-Za-z0-9]+', m['text'].upper()))
        overlap = len(qwords & mwords)/max(len(qwords),1)
        score = 0.4*cosine(q['fp'], m['fp']) + 0.6*overlap
        scored.append((score, m))
    scored.sort(key=lambda x: -x[0])
    top = scored[:top_k]
    q_morse = decode_morse(encode_morse(question))
    body = [f"[{m['source']} | {s:.2f}] {m['text']}" for s,m in top]
    answer = (
        f"QUERY (Morse-exact): {q_morse}\n"
        f"ARTICULATION:\n  {top[0][1]['text']}\n"
        f"EVIDENCE:\n" + "\n".join('  • '+b for b in body) +
        f"\nRESERVOIR: S̄={q['S_mean']:.3f} rate={q['rate']:.1f}Hz"
    )
    return answer, top

for q in [
    'What is Fluid Spacetime Omni-Theory?',
    'How does Morse trinary encode language?',
    'What are zero free parameters?',
    'How do EEG emotion labels map to FSOT drive?',
]:
    print('\n' + '='*60)
    ans, _ = query(q)
    print(ans)

## 4. Optional: full package path (if you attached FSOT code dataset)

If import works, run richer emotion integration + failure probe.

In [ ]:
if USE_PACKAGE:
    try:
        from fsot_nuron.emotions_eeg import compute_emotions_stats
        # point loader at Kaggle CSV
        import fsot_nuron.emotions_eeg as ee
        ee.CANDIDATES = [EMO_CSV] + list(ee.CANDIDATES)
        st = compute_emotions_stats(EMO_CSV)
        print('package stats labels', st.get('labels'))
        print('templates', st.get('emotion_lesion_templates'))
    except Exception as e:
        print('package path failed:', e)
else:
    print('Skip — attach fsot-2-1-neural-kaggle.zip as a second dataset for full package.')

## 5. Run other industry-standard Kaggle datasets the same way

1. **Add Data** → pick another EEG / text dataset  
2. Set `OTHER = Path('/kaggle/input/<slug>/...')`  
3. If CSV with labels: compute energy CV / class stats → FSOT templates  
4. If text: append rows into `CORPUS` and re-run chew/query  

Examples to try next (attach separately):
- `birdy654/eeg-brainwave-dataset-mental-state`
- `wanghaohan/confused-eeg`
- any public domain FSOT-related text corpus you own  

### Submit / share
- Save Version → **Save & Run All** on Kaggle  
- License notebook: Apache-2.0  
- Credit third-party data in the header (as above)  
- Point theory pin to FSOT-2.1-Lean on GitHub

In [ ]:
# Summary artifact for the run
summary = {
    'on_kaggle': ON_KAGGLE,
    'emotions_csv': str(EMO_CSV),
    'n_samples': int(df.shape[0]),
    'labels': y.value_counts().to_dict(),
    'energy_cv': energy_cv,
    'morse_roundtrip_ok': decode_morse(encode_morse('FSOT 21'))=='FSOT 21',
    'memory_chunks': len(memory),
    'package_used': USE_PACKAGE,
    'theory': 'https://github.com/dappalumbo91/FSOT-2.1-Lean',
    'dataset_input': 'birdy654/eeg-brainwave-dataset-feeling-emotions',
}
outp = WORK / 'fsot_kaggle_run_summary.json'
outp.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
print('wrote', outp)